# Wikidata Post-Import Verification and Repair

Created by [Matt Artz](https://www.mattartz.me/) — Advancing AI Anthropology through computational approaches to qualitative research.

---

## What This Notebook Does

After importing scholarly articles to Wikidata via QuickStatements, some records may fail completely or be created with missing properties. This notebook verifies what was actually imported and generates repair QuickStatements to fill in the gaps.

The notebook takes your original CrossRef CSV (the source of truth) and compares it against what actually exists in Wikidata's scholarly endpoint. It identifies:

1. **Missing Articles**: DOIs that exist in your CSV but not in Wikidata at all
2. **Missing Properties**: Articles that exist but are missing labels, titles, dates, authors, volume, issue, pages, or other expected fields

It then generates targeted QuickStatements to add only what's missing—whether that's complete new records or individual property additions to existing items.

## Key Features

- **Scholarly Endpoint**: Uses `query-scholarly.wikidata.org` (required since May 2025 graph split)
- **Field-by-Field Comparison**: Checks each property individually, not just existence
- **Incremental Repair**: Only generates statements for what's actually missing
- **Detailed Reporting**: Shows exactly what's missing where for manual review
- **Batch-Friendly Output**: Generates clean QuickStatements V1 format

## Workflow

1. **Upload Files**: Your CrossRef CSV and (optionally) a journal mapping CSV
2. **Set Journal Q-ID**: Specify the journal's Wikidata identifier
3. **Run Verification**: Query Wikidata for each DOI and compare properties
4. **Review Report**: See missing articles and missing properties
5. **Generate Repairs**: Create QuickStatements for missing items and values

## Citation

If you use this notebook, please cite:

> Artz, Matt. (2026). MattArtzAnthro/wikidata-tools. Zenodo. https://doi.org/10.5281/zenodo.18912858

## License

[CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)

## Setup

In [ ]:
# Install required packages
!pip install requests pandas ipywidgets -q

import requests
import pandas as pd
import re
import time
import json
from datetime import datetime
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
from io import BytesIO
import os

print("Setup complete.")

## Configuration

**CRITICAL**: Since May 2025, scholarly articles are ONLY on `query-scholarly.wikidata.org`. The main endpoint no longer contains scholarly articles!

In [ ]:
# IMPORTANT: Since May 2025 graph split, scholarly articles are ONLY on the scholarly endpoint
SCHOLARLY_ENDPOINT = "https://query-scholarly.wikidata.org/sparql"

# Properties we check and potentially repair
WIKIDATA_PROPERTIES = {
    'label_en': 'Len',           # English label
    'description_en': 'Den',      # English description
    'instance_of': 'P31',         # Should be Q13442814 (scholarly article)
    'title': 'P1476',             # Title (monolingual text)
    'doi': 'P356',                # DOI
    'publication_date': 'P577',   # Publication date
    'published_in': 'P1433',      # Journal
    'volume': 'P478',             # Volume
    'issue': 'P433',              # Issue number
    'pages': 'P304',              # Page range
    'issn': 'P236',               # ISSN
    'full_work_url': 'P953',      # Full work available at URL
    'language': 'P407',           # Language of work
    'author_name_string': 'P2093' # Author name string (can have multiple)
}

# User-Agent for API requests
USER_AGENT = "WikidataPostImportVerifier/1.0 (https://www.mattartz.me/)"

print(f"Using scholarly endpoint: {SCHOLARLY_ENDPOINT}")
print(f"Properties to verify: {len(WIKIDATA_PROPERTIES)}")

## Helper Functions

In [ ]:
def clean_doi(doi_input):
    """Normalize DOI to standard format."""
    if not doi_input or not isinstance(doi_input, str):
        return None
    doi_input = str(doi_input).strip()
    # Extract DOI from URLs
    for pattern in [r'https?://(?:dx\.)?doi\.org/(.+)', r'doi:(.+)']:
        match = re.search(pattern, doi_input, re.IGNORECASE)
        if match:
            doi_input = match.group(1)
            break
    # Validate DOI format
    if re.match(r'^10\.\d+/.+', doi_input):
        return doi_input.strip()
    return None

def doi_to_upper(doi):
    """Convert DOI to uppercase for Wikidata matching."""
    return doi.upper() if doi else None

def parse_authors(authors_string):
    """Parse semicolon-separated authors string."""
    if not authors_string or pd.isna(authors_string):
        return []
    return [a.strip() for a in str(authors_string).split(';') if a.strip()]

def parse_publication_date(date_string, year_string=None):
    """Parse date into Wikidata time format with precision."""
    if not date_string or pd.isna(date_string):
        if year_string and not pd.isna(year_string):
            return f"+{int(year_string)}-00-00T00:00:00Z", 9
        return None, None
    date_str = str(date_string).strip()
    if re.match(r'^\d{4}-\d{2}-\d{2}$', date_str):
        return f"+{date_str}T00:00:00Z", 11
    if re.match(r'^\d{4}-\d{2}$', date_str):
        return f"+{date_str}-00T00:00:00Z", 10
    if re.match(r'^\d{4}$', date_str):
        return f"+{date_str}-00-00T00:00:00Z", 9
    if year_string and not pd.isna(year_string):
        return f"+{int(year_string)}-00-00T00:00:00Z", 9
    return None, None

def get_first_issn(issn_string):
    """Get first ISSN from string that may contain multiple."""
    if not issn_string or pd.isna(issn_string):
        return None
    issns = re.split(r'[;,]', str(issn_string))
    return issns[0].strip() if issns else None

def escape_quickstatements(text):
    """Escape text for QuickStatements format."""
    if not text:
        return ""
    text = str(text).strip()
    # Escape quotes
    text = text.replace('"', '\\"')
    return text

print("Helper functions loaded.")

## Wikidata Query Functions

In [ ]:
def query_wikidata_by_doi(doi, verbose=False):
    """
    Query Wikidata scholarly endpoint for an article by DOI.
    Returns QID if found, None otherwise.
    """
    doi_upper = doi.upper()
    sparql_query = f'SELECT ?item WHERE {{ ?item wdt:P356 "{doi_upper}" }}'
    
    try:
        response = requests.get(
            SCHOLARLY_ENDPOINT,
            params={"query": sparql_query, "format": "json"},
            headers={"User-Agent": USER_AGENT},
            timeout=30
        )
        response.raise_for_status()
        data = response.json()
        results = data.get("results", {}).get("bindings", [])
        
        if results:
            qid = results[0]["item"]["value"].split("/")[-1]
            if verbose:
                print(f"  Found: {qid}")
            return qid
        return None
    except Exception as e:
        if verbose:
            print(f"  Error: {e}")
        return None

def get_article_properties(qid, verbose=False):
    """
    Get all relevant properties for an article from Wikidata.
    Returns dict of property values.
    """
    sparql_query = f"""
    SELECT ?item ?itemLabel ?itemDescription
           ?title ?titleLang
           ?doi ?pubDate ?journal ?journalLabel
           ?volume ?issue ?pages ?issn ?url ?language ?languageLabel
           (GROUP_CONCAT(DISTINCT ?authorName; SEPARATOR="||") AS ?authors)
    WHERE {{
      BIND(wd:{qid} AS ?item)
      
      OPTIONAL {{ ?item wdt:P1476 ?title. BIND(LANG(?title) AS ?titleLang) }}
      OPTIONAL {{ ?item wdt:P356 ?doi. }}
      OPTIONAL {{ ?item wdt:P577 ?pubDate. }}
      OPTIONAL {{ ?item wdt:P1433 ?journal. }}
      OPTIONAL {{ ?item wdt:P478 ?volume. }}
      OPTIONAL {{ ?item wdt:P433 ?issue. }}
      OPTIONAL {{ ?item wdt:P304 ?pages. }}
      OPTIONAL {{ ?item wdt:P236 ?issn. }}
      OPTIONAL {{ ?item wdt:P953 ?url. }}
      OPTIONAL {{ ?item wdt:P407 ?language. }}
      OPTIONAL {{ ?item wdt:P2093 ?authorName. }}
      
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    GROUP BY ?item ?itemLabel ?itemDescription ?title ?titleLang ?doi ?pubDate ?journal ?journalLabel ?volume ?issue ?pages ?issn ?url ?language ?languageLabel
    """
    
    try:
        response = requests.get(
            SCHOLARLY_ENDPOINT,
            params={"query": sparql_query, "format": "json"},
            headers={"User-Agent": USER_AGENT},
            timeout=60
        )
        response.raise_for_status()
        data = response.json()
        results = data.get("results", {}).get("bindings", [])
        
        if not results:
            return None
        
        r = results[0]
        props = {
            'qid': qid,
            'label_en': r.get('itemLabel', {}).get('value'),
            'description_en': r.get('itemDescription', {}).get('value'),
            'title': r.get('title', {}).get('value'),
            'title_lang': r.get('titleLang', {}).get('value'),
            'doi': r.get('doi', {}).get('value'),
            'publication_date': r.get('pubDate', {}).get('value'),
            'published_in_qid': r.get('journal', {}).get('value', '').split('/')[-1] if r.get('journal') else None,
            'published_in_label': r.get('journalLabel', {}).get('value'),
            'volume': r.get('volume', {}).get('value'),
            'issue': r.get('issue', {}).get('value'),
            'pages': r.get('pages', {}).get('value'),
            'issn': r.get('issn', {}).get('value'),
            'url': r.get('url', {}).get('value'),
            'language_qid': r.get('language', {}).get('value', '').split('/')[-1] if r.get('language') else None,
            'authors': r.get('authors', {}).get('value', '').split('||') if r.get('authors', {}).get('value') else []
        }
        
        # Clean up empty values
        props['authors'] = [a for a in props['authors'] if a]
        
        return props
        
    except Exception as e:
        if verbose:
            print(f"  Error getting properties for {qid}: {e}")
        return None

print("Query functions loaded.")

## Test Endpoint Connection

In [ ]:
# Test the scholarly endpoint with a known DOI
print("Testing scholarly endpoint connection...")
print()

test_doi = "10.1111/ANHU.12540"  # Known article
print(f"Testing with DOI: {test_doi}")

qid = query_wikidata_by_doi(test_doi, verbose=True)

if qid:
    print(f"\nSUCCESS: Found {qid}")
    print(f"View at: https://www.wikidata.org/wiki/{qid}")
    print("\nFetching properties...")
    props = get_article_properties(qid, verbose=True)
    if props:
        print(f"\nProperties retrieved:")
        for k, v in props.items():
            if v:
                print(f"  {k}: {v}")
else:
    print("NOT FOUND or endpoint issue")
    print("Check your network connection.")

## Upload Files

In [ ]:
# Global dataframe for uploaded CSV
articles_df = None

# File upload widgets
csv_upload = widgets.FileUpload(accept='.csv', multiple=False, description='CrossRef CSV')
csv_output = widgets.Output()

def handle_csv_upload(change):
    global articles_df
    with csv_output:
        clear_output()
        if csv_upload.value:
            file_info = list(csv_upload.value.values())[0]
            content = file_info['content']
            articles_df = pd.read_csv(BytesIO(content))
            print(f"✓ Loaded {len(articles_df)} rows from CSV")
            print(f"  Columns: {', '.join(articles_df.columns[:8])}...")
            
            # Count DOIs
            doi_col = 'DOI' if 'DOI' in articles_df.columns else 'doi'
            if doi_col in articles_df.columns:
                valid_dois = articles_df[doi_col].dropna().apply(clean_doi).dropna().count()
                print(f"  Valid DOIs: {valid_dois}")

csv_upload.observe(handle_csv_upload, names='value')

display(widgets.VBox([
    widgets.Label('Upload your CrossRef CSV:'),
    csv_upload,
    csv_output
]))

print("Upload your CrossRef CSV file above.")

## Configure Journal

In [ ]:
# Journal configuration
journal_qid_input = widgets.Text(
    value='',
    placeholder='e.g., Q15815513',
    description='Journal QID:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

journal_name_input = widgets.Text(
    value='',
    placeholder='e.g., American Anthropologist',
    description='Journal Name:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

journal_output = widgets.Output()

print("Enter the journal's Wikidata Q-ID (required for adding 'published in' property):")
print("You can find this by searching for the journal on wikidata.org")
print()
display(journal_qid_input)
display(journal_name_input)
display(journal_output)

## Run Verification

This cell queries Wikidata for each DOI and compares the properties against your source data.

In [ ]:
# Storage for verification results
verification_results = {
    'missing_articles': [],      # DOIs not in Wikidata at all
    'complete_articles': [],     # Articles with all expected properties
    'incomplete_articles': [],   # Articles missing some properties
    'source_data': {}            # Original CSV data keyed by DOI
}

verify_button = widgets.Button(description='Run Verification', button_style='primary')
progress_bar = widgets.IntProgress(value=0, min=0, max=100, description='Progress:')
verify_output = widgets.Output()

def run_verification(button):
    global verification_results
    
    verification_results = {
        'missing_articles': [],
        'complete_articles': [],
        'incomplete_articles': [],
        'source_data': {}
    }
    
    with verify_output:
        clear_output()
        
        if articles_df is None:
            print("❌ Please upload a CSV file first.")
            return
        
        journal_qid = journal_qid_input.value.strip()
        if not journal_qid:
            print("⚠️ No journal Q-ID specified. Will not check 'published in' property.")
        else:
            print(f"Journal Q-ID: {journal_qid}")
        print()
        
        # Identify DOI column
        doi_col = 'DOI' if 'DOI' in articles_df.columns else 'doi'
        if doi_col not in articles_df.columns:
            print("❌ No DOI column found in CSV.")
            return
        
        total = len(articles_df)
        progress_bar.max = total
        print(f"Verifying {total} articles against Wikidata...")
        print(f"Endpoint: {SCHOLARLY_ENDPOINT}")
        print("=" * 60)
        print()
        
        for idx, row in articles_df.iterrows():
            progress_bar.value = idx + 1
            
            raw_doi = row.get(doi_col, '')
            doi = clean_doi(raw_doi)
            title = str(row.get('Title', ''))[:50]
            
            if not doi:
                print(f"[{idx+1}] SKIP (no DOI): {title}...")
                continue
            
            # Store source data
            source_record = {
                'title': row.get('Title', ''),
                'doi': doi,
                'authors': parse_authors(row.get('Authors', '')),
                'publication_date': row.get('Publication Date', ''),
                'year': row.get('Year', ''),
                'volume': str(row.get('Volume', '')) if pd.notna(row.get('Volume')) else None,
                'issue': str(row.get('Issue', '')) if pd.notna(row.get('Issue')) else None,
                'pages': str(row.get('Page', '')) if pd.notna(row.get('Page')) else None,
                'issn': get_first_issn(row.get('ISSN', '')),
                'url': row.get('URL', ''),
                'language': row.get('Language', 'en'),
                'journal': row.get('Journal', ''),
                'journal_qid': journal_qid
            }
            verification_results['source_data'][doi.upper()] = source_record
            
            # Check if article exists in Wikidata
            qid = query_wikidata_by_doi(doi)
            
            if not qid:
                # Article not found - completely missing
                verification_results['missing_articles'].append({
                    'doi': doi,
                    'source': source_record
                })
                print(f"[{idx+1}] MISSING: {title}...")
            else:
                # Article exists - check properties
                wikidata_props = get_article_properties(qid)
                
                if not wikidata_props:
                    print(f"[{idx+1}] ERROR getting props for {qid}: {title}...")
                    continue
                
                # Compare properties
                missing_props = []
                
                # Check label
                if not wikidata_props.get('label_en') or wikidata_props['label_en'] == qid:
                    missing_props.append('label_en')
                
                # Check description
                if not wikidata_props.get('description_en'):
                    missing_props.append('description_en')
                
                # Check title (P1476)
                if not wikidata_props.get('title'):
                    missing_props.append('title')
                
                # Check publication date (P577)
                if not wikidata_props.get('publication_date') and source_record.get('publication_date'):
                    missing_props.append('publication_date')
                
                # Check published in (P1433)
                if journal_qid and not wikidata_props.get('published_in_qid'):
                    missing_props.append('published_in')
                
                # Check volume (P478)
                if not wikidata_props.get('volume') and source_record.get('volume'):
                    missing_props.append('volume')
                
                # Check issue (P433)
                if not wikidata_props.get('issue') and source_record.get('issue'):
                    missing_props.append('issue')
                
                # Check pages (P304)
                if not wikidata_props.get('pages') and source_record.get('pages'):
                    missing_props.append('pages')
                
                # Check ISSN (P236)
                if not wikidata_props.get('issn') and source_record.get('issn'):
                    missing_props.append('issn')
                
                # Check URL (P953)
                if not wikidata_props.get('url') and source_record.get('url'):
                    missing_props.append('url')
                
                # Check language (P407)
                if not wikidata_props.get('language_qid'):
                    missing_props.append('language')
                
                # Check authors (P2093)
                if not wikidata_props.get('authors') and source_record.get('authors'):
                    missing_props.append('authors')
                
                if missing_props:
                    verification_results['incomplete_articles'].append({
                        'qid': qid,
                        'doi': doi,
                        'missing_properties': missing_props,
                        'wikidata': wikidata_props,
                        'source': source_record
                    })
                    print(f"[{idx+1}] INCOMPLETE ({qid}): Missing {', '.join(missing_props[:3])}{'...' if len(missing_props) > 3 else ''}")
                else:
                    verification_results['complete_articles'].append({
                        'qid': qid,
                        'doi': doi
                    })
                    print(f"[{idx+1}] COMPLETE ({qid}): {title[:30]}...")
            
            # Rate limiting
            time.sleep(0.5)
        
        # Summary
        print()
        print("=" * 60)
        print("VERIFICATION SUMMARY")
        print("=" * 60)
        print(f"Total articles checked:    {len(verification_results['source_data'])}")
        print(f"Complete in Wikidata:      {len(verification_results['complete_articles'])}")
        print(f"Missing from Wikidata:     {len(verification_results['missing_articles'])}")
        print(f"Incomplete in Wikidata:    {len(verification_results['incomplete_articles'])}")
        
        if verification_results['incomplete_articles']:
            print()
            print("Most common missing properties:")
            prop_counts = {}
            for article in verification_results['incomplete_articles']:
                for prop in article['missing_properties']:
                    prop_counts[prop] = prop_counts.get(prop, 0) + 1
            for prop, count in sorted(prop_counts.items(), key=lambda x: -x[1])[:10]:
                print(f"  {prop}: {count}")

verify_button.on_click(run_verification)
display(widgets.VBox([verify_button, progress_bar, verify_output]))

## Detailed Report

View details of missing and incomplete articles.

In [ ]:
report_output = widgets.Output()

with report_output:
    clear_output()
    
    if not verification_results.get('missing_articles') and not verification_results.get('incomplete_articles'):
        print("Run verification first to see the report.")
    else:
        # Missing articles
        if verification_results['missing_articles']:
            print("=" * 70)
            print(f"MISSING ARTICLES ({len(verification_results['missing_articles'])} total)")
            print("These DOIs were not found in Wikidata at all.")
            print("=" * 70)
            for i, article in enumerate(verification_results['missing_articles'][:20], 1):
                print(f"\n{i}. {article['source']['title'][:70]}")
                print(f"   DOI: {article['doi']}")
                print(f"   Authors: {', '.join(article['source']['authors'][:3])}" + 
                      ("..." if len(article['source']['authors']) > 3 else ""))
            if len(verification_results['missing_articles']) > 20:
                print(f"\n... and {len(verification_results['missing_articles']) - 20} more")
        
        # Incomplete articles
        if verification_results['incomplete_articles']:
            print("\n")
            print("=" * 70)
            print(f"INCOMPLETE ARTICLES ({len(verification_results['incomplete_articles'])} total)")
            print("These articles exist but are missing properties.")
            print("=" * 70)
            for i, article in enumerate(verification_results['incomplete_articles'][:20], 1):
                title = article['source']['title'][:50] or article['wikidata'].get('label_en', 'No title')[:50]
                print(f"\n{i}. {title}...")
                print(f"   QID: {article['qid']} | DOI: {article['doi']}")
                print(f"   Missing: {', '.join(article['missing_properties'])}")
            if len(verification_results['incomplete_articles']) > 20:
                print(f"\n... and {len(verification_results['incomplete_articles']) - 20} more")

display(report_output)

# Button to refresh report
refresh_button = widgets.Button(description='Refresh Report', button_style='info')
def refresh_report(b):
    with report_output:
        clear_output()
        if not verification_results.get('missing_articles') and not verification_results.get('incomplete_articles'):
            print("Run verification first.")
            return
        # Same report code...
        if verification_results['missing_articles']:
            print(f"MISSING ARTICLES: {len(verification_results['missing_articles'])}")
        if verification_results['incomplete_articles']:
            print(f"INCOMPLETE ARTICLES: {len(verification_results['incomplete_articles'])}")
        print("\n(Full report shown above)")
refresh_button.on_click(refresh_report)
display(refresh_button)

## Generate Repair QuickStatements

This generates two types of QuickStatements:
1. **CREATE statements** for completely missing articles
2. **QID-targeted statements** for adding missing properties to existing articles

In [ ]:
def generate_new_article_qs(source):
    """Generate QuickStatements for a completely new article."""
    lines = []
    
    title = escape_quickstatements(source.get('title', ''))
    doi = source.get('doi', '').upper()
    journal_qid = source.get('journal_qid', '')
    journal_name = source.get('journal', '')
    
    # Parse date
    time_str, precision = parse_publication_date(
        source.get('publication_date'), 
        source.get('year')
    )
    year = source.get('year', '')
    if not year and source.get('publication_date'):
        year_match = re.match(r'(\d{4})', str(source.get('publication_date')))
        if year_match:
            year = year_match.group(1)
    
    # Get volume, issue for description
    vol_str = str(source.get('volume', '')).strip() if source.get('volume') else ''
    iss_str = str(source.get('issue', '')).strip() if source.get('issue') else ''
    
    # CREATE
    lines.append("CREATE")
    
    # Label (Len)
    if title:
        lines.append(f'LAST|Len|"{title}"')
    
    # Description (Den) - include vol/issue/DOI for uniqueness
    if journal_name:
        desc = f"scholarly article published in {journal_name}"
        if vol_str and iss_str:
            desc += f", vol. {vol_str} no. {iss_str}"
        elif vol_str:
            desc += f", vol. {vol_str}"
        if year:
            desc += f" ({year})"
        if doi:
            desc += f" (DOI: {doi})"
    else:
        desc = f"scholarly article ({year})" if year else "scholarly article"
        if doi:
            desc += f" (DOI: {doi})"
    lines.append(f'LAST|Den|"{desc[:250]}"')
    
    # Instance of scholarly article (P31)
    lines.append('LAST|P31|Q13442814')
    
    # Title (P1476)
    if title:
        lang = source.get('language', 'en') or 'en'
        lines.append(f'LAST|P1476|{lang}:"{title}"')
    
    # DOI (P356)
    if doi:
        lines.append(f'LAST|P356|"{doi}"')
    
    # Publication date (P577)
    if time_str:
        lines.append(f'LAST|P577|{time_str}/{precision}')
    
    # Published in (P1433)
    if journal_qid:
        lines.append(f'LAST|P1433|{journal_qid}')
    
    # Authors (P2093)
    for i, author in enumerate(source.get('authors', []), 1):
        author_escaped = escape_quickstatements(author)
        lines.append(f'LAST|P2093|"{author_escaped}"|P1545|"{i}"')
    
    # Volume (P478)
    if source.get('volume'):
        lines.append(f'LAST|P478|"{source["volume"]}"')
    
    # Issue (P433)
    if source.get('issue'):
        lines.append(f'LAST|P433|"{source["issue"]}"')
    
    # Pages (P304)
    if source.get('pages'):
        lines.append(f'LAST|P304|"{source["pages"]}"')
    
    # ISSN (P236)
    if source.get('issn'):
        lines.append(f'LAST|P236|"{source["issn"]}"')
    
    # Full work URL (P953)
    if source.get('url'):
        lines.append(f'LAST|P953|"{source["url"]}"')
    
    # Language (P407) - Q1860 is English
    lang = source.get('language', 'en')
    if lang == 'en':
        lines.append('LAST|P407|Q1860')
    
    return lines

def generate_property_repair_qs(article):
    """Generate QuickStatements to add missing properties to an existing article."""
    lines = []
    qid = article['qid']
    source = article['source']
    missing = article['missing_properties']
    
    title = escape_quickstatements(source.get('title', ''))
    doi = source.get('doi', '').upper()
    
    # Parse date
    time_str, precision = parse_publication_date(
        source.get('publication_date'), 
        source.get('year')
    )
    year = source.get('year', '')
    if not year and source.get('publication_date'):
        year_match = re.match(r'(\d{4})', str(source.get('publication_date')))
        if year_match:
            year = year_match.group(1)
    
    journal_qid = source.get('journal_qid', '')
    journal_name = source.get('journal', '')
    
    # Get volume, issue for description
    vol_str = str(source.get('volume', '')).strip() if source.get('volume') else ''
    iss_str = str(source.get('issue', '')).strip() if source.get('issue') else ''
    
    # Label
    if 'label_en' in missing and title:
        lines.append(f'{qid}|Len|"{title}"')
    
    # Description - include vol/issue/DOI for uniqueness
    if 'description_en' in missing:
        if journal_name:
            desc = f"scholarly article published in {journal_name}"
            if vol_str and iss_str:
                desc += f", vol. {vol_str} no. {iss_str}"
            elif vol_str:
                desc += f", vol. {vol_str}"
            if year:
                desc += f" ({year})"
            if doi:
                desc += f" (DOI: {doi})"
        else:
            desc = f"scholarly article ({year})" if year else "scholarly article"
            if doi:
                desc += f" (DOI: {doi})"
        lines.append(f'{qid}|Den|"{desc[:250]}"')
    
    # Title (P1476)
    if 'title' in missing and title:
        lang = source.get('language', 'en') or 'en'
        lines.append(f'{qid}|P1476|{lang}:"{title}"')
    
    # Publication date (P577)
    if 'publication_date' in missing and time_str:
        lines.append(f'{qid}|P577|{time_str}/{precision}')
    
    # Published in (P1433)
    if 'published_in' in missing and journal_qid:
        lines.append(f'{qid}|P1433|{journal_qid}')
    
    # Authors (P2093)
    if 'authors' in missing:
        for i, author in enumerate(source.get('authors', []), 1):
            author_escaped = escape_quickstatements(author)
            lines.append(f'{qid}|P2093|"{author_escaped}"|P1545|"{i}"')
    
    # Volume (P478)
    if 'volume' in missing and source.get('volume'):
        lines.append(f'{qid}|P478|"{source["volume"]}"')
    
    # Issue (P433)
    if 'issue' in missing and source.get('issue'):
        lines.append(f'{qid}|P433|"{source["issue"]}"')
    
    # Pages (P304)
    if 'pages' in missing and source.get('pages'):
        lines.append(f'{qid}|P304|"{source["pages"]}"')
    
    # ISSN (P236)
    if 'issn' in missing and source.get('issn'):
        lines.append(f'{qid}|P236|"{source["issn"]}"')
    
    # Full work URL (P953)
    if 'url' in missing and source.get('url'):
        lines.append(f'{qid}|P953|"{source["url"]}"')
    
    # Language (P407)
    if 'language' in missing:
        lang = source.get('language', 'en')
        if lang == 'en':
            lines.append(f'{qid}|P407|Q1860')
    
    return lines

print("QuickStatements generator functions loaded.")
print("Descriptions include vol/issue/DOI for uniqueness.")


In [ ]:
# Generation options
gen_missing = widgets.Checkbox(value=True, description='Generate CREATE statements for missing articles')
gen_repairs = widgets.Checkbox(value=True, description='Generate repair statements for incomplete articles')

generate_button = widgets.Button(description='Generate Repair QuickStatements', button_style='success')
generate_output = widgets.Output()

def generate_repairs(button):
    with generate_output:
        clear_output()
        
        if not verification_results.get('missing_articles') and not verification_results.get('incomplete_articles'):
            print("❌ Run verification first.")
            return
        
        all_lines = []
        
        # Generate CREATE statements for missing articles
        if gen_missing.value and verification_results['missing_articles']:
            print(f"Generating CREATE statements for {len(verification_results['missing_articles'])} missing articles...")
            for article in verification_results['missing_articles']:
                lines = generate_new_article_qs(article['source'])
                all_lines.extend(lines)
                all_lines.append('')  # Blank line between articles
        
        # Generate repair statements for incomplete articles
        if gen_repairs.value and verification_results['incomplete_articles']:
            print(f"Generating repair statements for {len(verification_results['incomplete_articles'])} incomplete articles...")
            for article in verification_results['incomplete_articles']:
                lines = generate_property_repair_qs(article)
                if lines:
                    all_lines.extend(lines)
                    all_lines.append('')  # Blank line between articles
        
        if not all_lines:
            print("No QuickStatements to generate.")
            return
        
        # Join and save
        qs_text = '\n'.join(all_lines)
        
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"repair_quickstatements_{timestamp}.txt"
        
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(qs_text)
        
        print()
        print("=" * 60)
        print("GENERATION COMPLETE")
        print("=" * 60)
        print(f"File saved: {filename}")
        print(f"Total lines: {len([l for l in all_lines if l])}")
        print(f"CREATE blocks: {qs_text.count('CREATE')}")
        print(f"Repair statements: {len([l for l in all_lines if l and not l.startswith('CREATE') and not l.startswith('LAST')])}")
        print()
        
        # Preview
        print("--- PREVIEW (first 30 lines) ---")
        preview_lines = [l for l in all_lines if l][:30]
        for line in preview_lines:
            print(line)
        if len(all_lines) > 30:
            print("...")
        
        print()
        print("--- UPLOAD INSTRUCTIONS ---")
        print("1. Go to: https://quickstatements.toolforge.org/")
        print("2. Log in with your Wikidata account")
        print("3. Click 'New batch'")
        print("4. Paste the file contents (or upload the file)")
        print("5. Click 'Import V1 commands'")
        print("6. Review and click 'Run'")
        print()
        print("TIP: For large files, use the split_txt_quickstatements notebook to create smaller batches.")
        
        # Download
        try:
            from google.colab import files
            files.download(filename)
        except:
            pass

generate_button.on_click(generate_repairs)
display(widgets.VBox([gen_missing, gen_repairs, generate_button, generate_output]))

## Export Verification Report

Export a detailed CSV report of all verification results.

In [ ]:
export_button = widgets.Button(description='Export Report CSV', button_style='info')
export_output = widgets.Output()

def export_report(button):
    with export_output:
        clear_output()
        
        rows = []
        
        # Complete articles
        for article in verification_results.get('complete_articles', []):
            rows.append({
                'Status': 'COMPLETE',
                'QID': article['qid'],
                'DOI': article['doi'],
                'Missing_Properties': '',
                'Title': verification_results['source_data'].get(article['doi'].upper(), {}).get('title', '')
            })
        
        # Incomplete articles
        for article in verification_results.get('incomplete_articles', []):
            rows.append({
                'Status': 'INCOMPLETE',
                'QID': article['qid'],
                'DOI': article['doi'],
                'Missing_Properties': ', '.join(article['missing_properties']),
                'Title': article['source'].get('title', '')
            })
        
        # Missing articles
        for article in verification_results.get('missing_articles', []):
            rows.append({
                'Status': 'MISSING',
                'QID': '',
                'DOI': article['doi'],
                'Missing_Properties': 'ALL',
                'Title': article['source'].get('title', '')
            })
        
        if not rows:
            print("No results to export. Run verification first.")
            return
        
        df = pd.DataFrame(rows)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"verification_report_{timestamp}.csv"
        df.to_csv(filename, index=False)
        
        print(f"Exported: {filename}")
        print(f"Total rows: {len(rows)}")
        print(f"  COMPLETE: {len([r for r in rows if r['Status'] == 'COMPLETE'])}")
        print(f"  INCOMPLETE: {len([r for r in rows if r['Status'] == 'INCOMPLETE'])}")
        print(f"  MISSING: {len([r for r in rows if r['Status'] == 'MISSING'])}")
        
        try:
            from google.colab import files
            files.download(filename)
        except:
            pass

export_button.on_click(export_report)
display(widgets.VBox([export_button, export_output]))

## Generate Separate Files

Optionally generate separate QuickStatements files for:
1. New articles (CREATE statements)
2. Property repairs (targeted additions to existing items)

In [ ]:
split_button = widgets.Button(description='Generate Separate Files', button_style='warning')
split_output = widgets.Output()

def generate_split_files(button):
    with split_output:
        clear_output()
        
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        
        # File 1: New articles
        if verification_results.get('missing_articles'):
            new_lines = []
            for article in verification_results['missing_articles']:
                lines = generate_new_article_qs(article['source'])
                new_lines.extend(lines)
                new_lines.append('')
            
            filename1 = f"new_articles_{timestamp}.txt"
            with open(filename1, 'w', encoding='utf-8') as f:
                f.write('\n'.join(new_lines))
            print(f"✓ Created: {filename1}")
            print(f"  {len(verification_results['missing_articles'])} new articles")
            print(f"  {new_lines.count('CREATE')} CREATE blocks")
            print()
            
            try:
                from google.colab import files
                files.download(filename1)
            except:
                pass
        else:
            print("No missing articles to create.")
            print()
        
        # File 2: Property repairs
        if verification_results.get('incomplete_articles'):
            repair_lines = []
            for article in verification_results['incomplete_articles']:
                lines = generate_property_repair_qs(article)
                if lines:
                    repair_lines.extend(lines)
                    repair_lines.append('')
            
            if repair_lines:
                filename2 = f"property_repairs_{timestamp}.txt"
                with open(filename2, 'w', encoding='utf-8') as f:
                    f.write('\n'.join(repair_lines))
                print(f"✓ Created: {filename2}")
                print(f"  {len(verification_results['incomplete_articles'])} articles to repair")
                print(f"  {len([l for l in repair_lines if l])} property statements")
                
                try:
                    from google.colab import files
                    files.download(filename2)
                except:
                    pass
        else:
            print("No incomplete articles to repair.")

split_button.on_click(generate_split_files)
display(widgets.VBox([split_button, split_output]))

## Tips for Reducing QuickStatements Failures

Based on common failure patterns:

### Why QuickStatements Fail

1. **Rate Limiting**: Too many statements too fast. QuickStatements has internal rate limits.
2. **Character Encoding**: Special characters (em-dashes, curly quotes, accented characters) can cause parsing errors.
3. **Long Titles**: Titles over 250 characters may fail.
4. **Duplicate DOIs**: If an article was created between your initial check and upload.
5. **Edit Conflicts**: Multiple edits to the same item in quick succession.
6. **Server Timeouts**: Wikidata can be slow during peak hours.

### Best Practices

1. **Split into smaller batches**: Use `split_txt_quickstatements.ipynb` to create 50-100 statement batches.
2. **Run during off-peak hours**: European nighttime (UTC 22:00-06:00) tends to be faster.
3. **Check batch status**: Monitor the QuickStatements batch page for errors.
4. **Re-run verification after each batch**: This notebook helps you find what's still missing.
5. **Use the "Property repairs" file**: Adding properties to existing items is more reliable than CREATE.